In [3]:
import pandas as pd

term = pd.read_csv('../../../etl//raw_data/교육부 국사편찬위원회_한국역사용어시소러스 정보_20211028 (1).csv')

event = pd.read_csv('../../../etl/raw_data/한국고전종합DB_관계망/itkc_events.csv')

event_rela = pd.read_csv('../../../etl/raw_data/한국고전종합DB_관계망/itkc_event_relations.csv')

In [4]:
event.head()

,scope,event_id,event_name,subject_category,period,event_date,person_count,related_event,detail_url
0,event_subject,ITKC_PH_1294A_0435,강동성전투,전쟁,고려,1218년(고종 5) 12월\r\n~ 1219년(고종 6) 1월,2,고려거란전쟁,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
1,event_period,ITKC_PH_1294A_0435,강동성전투,전쟁,고려,1218년(고종 5) 12월\r\n~ 1219년(고종 6) 1월,2,고려거란전쟁,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
2,event_period,ITKC_PH_1294A_0435,강동성전투,전쟁,고려,1218년(고종 5) 12월\r\n~ 1219년(고종 6) 1월,2,고려거란전쟁,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
3,event_period,ITKC_PH_1294A_0436,거란의 항복,전쟁,고려,1219년(고종 6) 1월,2,고려거란전쟁,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
4,event_period,ITKC_PH_1294A_0437,최충헌의 사망,정치인,고려,1219년(고종 6) 9월,4,최씨무인정권,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...


In [5]:
event_rela.head()

,scope,event_id,event_name,relation_type,person_id,person_name,related_event_id,related_event_name,evidence_url,detail_url
0,event_subject,ITKC_PH_1294A_0435,강동성전투,사건인물,P009535,김인경(金仁鏡),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
1,event_subject,ITKC_PH_1294A_0435,강동성전투,사건인물,P058283,조충(趙冲),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
2,event_period,ITKC_PH_1294A_0435,강동성전투,사건인물,P009535,김인경(金仁鏡),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
3,event_period,ITKC_PH_1294A_0435,강동성전투,사건인물,P058283,조충(趙冲),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
4,event_period,ITKC_PH_1294A_0435,강동성전투,사건인물,P009535,김인경(金仁鏡),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...


## 사전 설계 정리: 왜 각 dictionary/staging 파일이 필요한가

Neo4j 전처리에서 만드는 파일은 전부 같은 성격이 아니다. 크게 세 종류로 나눠서 봐야 한다.

| 구분 | 의미 | 예시 |
|---|---|---|
| Dictionary | 표준 목록을 정하는 파일 | `category_dictionary.csv`, `event_category_dictionary.csv`, `relation_type_dictionary.csv` |
| Staging relation | 원본 데이터를 Neo4j 관계로 넣기 좋게 펼친 중간 파일 | `term_category_relation.csv`, `event_category_relation.csv` |
| Mapping | 서로 다른 분류 체계를 연결하는 파일 | `category_mapping.csv` |

즉, 사전은 단순히 CSV를 하나 더 만드는 작업이 아니라, 원본의 애매한 문자열을 그래프에서 재사용 가능한 기준으로 바꾸는 작업이다.

### 1. `category_dictionary.csv`

`history_terms.term_lk`에서 만든 표준 카테고리 사전이다.

원본 `term_lk`는 다음처럼 문자열 하나에 계층 정보가 들어 있다.

```text
정치·행정·법제>행정>중앙행정기구
교통·통신>교통시설>>교통·통신>교통로
```

여기서 `>`는 계층이고, `>>`는 복수 카테고리 경로다. 이 값을 그대로 `Term` 속성에만 넣으면 상위/하위 카테고리 탐색이 어렵다. 그래서 각 계층을 `Category` 노드로 만들기 위한 사전이 필요하다.

이 사전이 필요한 이유는 다음과 같다.

- `Category` 노드의 기준 목록이 된다.
- `SUBCATEGORY_OF` 관계를 만들 수 있다.
- 같은 카테고리에 속한 용어를 찾을 수 있다.
- 문제 생성에서 같은 분류의 오답 후보를 찾을 수 있다.
- `event_category_dictionary`와 매핑할 기준점이 된다.

핵심 컬럼은 다음과 같다.

| 컬럼 | 의미 |
|---|---|
| `category_id` | 카테고리 고유 ID |
| `category_name` | 현재 단계 이름 |
| `category_path` | 루트부터 현재 단계까지의 전체 경로 |
| `parent_category_id` | 상위 카테고리 ID |
| `parent_category_path` | 상위 카테고리 경로 |
| `depth` | 계층 깊이 |
| `root_category_name` | 최상위 카테고리 이름 |
| `term_count` | 해당 경로에 연결되는 용어 수 |
| `review_status` | 검수 상태 |

### 2. `term_category_relation.csv`

이 파일은 엄밀히 말하면 dictionary가 아니라 staging relation이다. `Term`과 `Category`를 연결하기 위해 만든 중간 산출물이다.

`category_dictionary.csv`만 있으면 카테고리 목록은 알 수 있지만, 어떤 `term_id`가 어떤 카테고리에 속하는지는 알 수 없다. 그 연결 정보가 `term_category_relation.csv`다.

예를 들어 원본이 다음과 같다면:

```text
term_id = 5626
term_lk = 문화·예술>음악
```

관계 staging은 다음처럼 된다.

```text
5626 -> 문화·예술>음악
```

`>>`가 있는 경우에는 한 용어가 여러 카테고리에 연결된다.

```text
교통·통신>교통시설>>교통·통신>교통로
```

위 값은 다음 두 관계로 펼쳐진다.

```text
Term -> 교통·통신>교통시설
Term -> 교통·통신>교통로
```

이 파일이 필요한 이유는 다음과 같다.

- Cypher에서 `>`, `>>` 파싱을 반복하지 않아도 된다.
- Neo4j import가 단순해진다.
- 원본 `term_lk`에서 어떤 관계가 만들어졌는지 검수할 수 있다.
- 복수 카테고리 연결을 명확하게 확인할 수 있다.

다만 이 파일은 사전이 아니라 관계 생성용 staging이므로 위치는 `dictionary/`보다 `staging/`이 더 자연스럽다.

### 3. `event_category_dictionary.csv`

`itkc_events.csv.subject_category`에서 만든 이벤트 전용 카테고리 사전이다.

이벤트의 `subject_category`는 `history_terms.term_lk`와 성격이 다르다. 예를 들어 이벤트에는 다음 값들이 있다.

```text
전쟁
반란
옥사
고변/탄핵
반란,\r\n\r\n정치인
```

이 값들은 역사용어의 계층형 카테고리라기보다 사건 수집 과정에서 붙은 사건 분류에 가깝다. 그래서 `category_dictionary.csv`에 바로 합치면 의미가 섞일 수 있다.

이 사전이 필요한 이유는 다음과 같다.

- 이벤트 원본 분류를 보존한다.
- 복합 문자열을 토큰 단위로 정리한다.
- 이벤트 분류의 빈도와 검수 대상을 확인할 수 있다.
- 표준 카테고리와 직접 합치지 않고 매핑할 준비를 한다.

핵심 컬럼은 다음과 같다.

| 컬럼 | 의미 |
|---|---|
| `event_category_id` | 이벤트 카테고리 고유 ID |
| `event_category_name` | 분리된 이벤트 카테고리 이름 |
| `event_count` | 해당 카테고리를 가진 사건 수 |
| `source` | 원본 컬럼 |
| `review_status` | 검수 상태 |

### 4. `event_category_relation.csv`

이 파일도 dictionary가 아니라 staging relation이다. `Event`와 `EventCategory`를 연결하기 위한 파일이다.

이벤트 카테고리도 쿼리로 생성할 수는 있다. 하지만 `subject_category`에 쉼표와 줄바꿈이 섞여 있고, 같은 사건이 여러 scope에서 중복 수집된 구조라 전처리 단계에서 펼쳐두는 편이 안전하다.

이 파일이 필요한 이유는 다음과 같다.

- `Event - HAS_EVENT_CATEGORY - EventCategory` 관계를 명확히 만든다.
- `subject_category` 복합값을 여러 관계로 펼친다.
- 같은 `event_id` 중복을 정리한 뒤 관계를 생성할 수 있다.
- 원본 `subject_category`를 보존해 검수할 수 있다.

예시는 다음과 같다.

```text
event_id = ITKC_PH_1294A_0435
subject_category = 전쟁

Event -> 전쟁
```

복합값은 다음처럼 펼쳐진다.

```text
subject_category = 반란,\r\n\r\n정치인

Event -> 반란
Event -> 정치인
```

### 5. `category_mapping.csv`

이 파일은 `event_category_dictionary.csv`와 `category_dictionary.csv`를 연결하는 매핑표다. 두 사전을 대체하는 파일이 아니다.

두 사전은 계속 필요하다.

```text
event_category_dictionary.csv = 이벤트 원본 분류 사전
category_dictionary.csv = history_terms.term_lk 기반 표준 카테고리 사전
category_mapping.csv = 두 분류 체계를 연결하는 규칙표
```

이 파일이 필요한 이유는 두 데이터셋의 분류 체계가 다르기 때문이다.

예를 들어 이벤트 카테고리의 `전쟁`은 `history_terms`의 특정 경로와 완전히 같은 문자열이 아닐 수 있다. 따라서 직접 병합하지 않고 다음처럼 매핑해야 한다.

```text
전쟁 -> 국방·군사
옥사 -> 정치·행정·법제>사법
국왕 -> 인물
기관 -> 정치·행정·법제>행정>중앙행정기구
```

이 매핑표가 있으면 다음이 가능하다.

- Event와 Term을 공통 카테고리 기준으로 연결한다.
- 이벤트 기반 문제와 용어 기반 문제를 같은 분류 체계에서 다룬다.
- 매핑이 애매한 항목을 검수 대상으로 분리한다.
- 자동 매핑과 수동 매핑을 구분한다.

핵심 컬럼은 다음과 같다.

| 컬럼 | 의미 |
|---|---|
| `event_category_id` | 이벤트 카테고리 ID |
| `event_category_name` | 이벤트 카테고리 이름 |
| `mapped_category_id` | 표준 카테고리 ID |
| `mapped_category_path` | 표준 카테고리 경로 |
| `mapping_type` | `EXACT`, `PARTIAL`, `MANUAL`, `UNMAPPED` |
| `confidence` | 매핑 신뢰도 |
| `review_status` | 검수 상태 |
| `note` | 비고 |

### 6. `period_dictionary.csv`

시대명과 기간을 정규화하기 위한 사전이다.

원본에는 시대 정보가 여러 컬럼에 흩어져 있다.

```text
history_terms.term_times
history_terms.term_year
itkc_events.period
itkc_events.event_date
```

이 값들은 `고려`, `고려전기`, `조선후기`, `삼국시대-조선시대`, `1218년(고종 5) 12월`처럼 형태가 다르다. 그대로 두면 시대별 검색과 연도 범위 필터링이 어렵다.

이 사전이 필요한 이유는 다음과 같다.

- `Period` 노드의 기준 목록이 된다.
- 시대명을 표준화한다.
- `IN_PERIOD` 관계 생성 기준이 된다.
- 같은 시대 오답 후보를 찾을 수 있다.
- 연도 범위 검색과 시대 검색을 함께 쓸 수 있다.

핵심 컬럼은 다음과 같다.

| 컬럼 | 의미 |
|---|---|
| `period_id` | 시대 고유 ID |
| `period_name` | 표준 시대명 |
| `period_level` | 시대 계층 수준 |
| `start_year` | 시작 연도 |
| `end_year` | 종료 연도 |
| `source` | 생성 근거 |
| `review_status` | 검수 상태 |
| `note` | 비고 |

### 7. `event_date_parse.csv`

이 파일은 dictionary가 아니라 날짜 정규화 staging이다. `event_date` 원문에서 연도, 월, 왕대 표현을 뽑아 Event 노드 속성과 Period 연결에 쓰기 위한 파일이다.

예를 들어 다음 원문이 있다.

```text
1218년(고종 5) 12월\r\n~ 1219년(고종 6) 1월
```

여기서 최소한 다음 정보를 뽑을 수 있다.

```text
start_year = 1218
end_year = 1219
start_month = 12
end_month = 1
start_reign_name = 고종
start_reign_year = 5
end_reign_name = 고종
end_reign_year = 6
date_precision = YEAR_MONTH_RANGE
```

이 파일이 필요한 이유는 다음과 같다.

- 사건을 연도 기준으로 정렬할 수 있다.
- 같은 시기 사건을 찾을 수 있다.
- `Period` 연결의 보조 근거가 된다.
- 왕대 표현을 나중에 `Reign` 사전과 연결할 수 있다.
- 날짜 파싱 실패 항목을 검수 대상으로 분리할 수 있다.

지금 1차에서는 음력/양력 변환이나 왕대 연도 역산까지 할 필요는 없다. 원문 보존, 연도 추출, 범위 여부, 월 추출, 왕대 문자열 보존 정도가 현실적인 목표다.

### 8. `relation_type_dictionary.csv`

`itkc_person_relations.csv.relation_type`을 그래프에서 쓸 수 있는 의미 규칙으로 바꾸는 사전이다.

원본 관계 유형은 다음처럼 짧은 문자열이다.

```text
형제, 자, 부, 조부, 장인, 사위, 증조부, 교유, 스승, 제자, 생부, 출자, 아내, 남편, 모, 생모
```

사람은 `부`를 보면 아버지라는 것을 알지만, 코드와 DB는 다음 판단을 자동으로 알 수 없다.

- 관계 방향이 `person_id -> related_person_id`인지
- 대칭 관계인지
- 역관계가 무엇인지
- 가족 관계인지 사회 관계인지
- 동세대 관계인지 윗세대/아랫세대 관계인지
- 문제 생성이나 챗봇에서 어떤 그룹으로 묶어야 하는지

그래서 relation type 사전이 필요하다.

예시는 다음과 같다.

| raw_relation_type | normalized_relation_type | relation_group | direction_rule | is_symmetric | inverse_relation_type |
|---|---|---|---|---|---|
| `부` | `HAS_FATHER` | `FAMILY_PARENT` | `person_to_related` | `N` | `HAS_CHILD` |
| `자` | `HAS_CHILD` | `FAMILY_CHILD` | `person_to_related` | `N` | `HAS_PARENT` |
| `형제` | `SIBLING_OF` | `FAMILY_SIBLING` | `undirected` | `Y` | `SIBLING_OF` |
| `교유` | `ASSOCIATED_WITH` | `SOCIAL` | `undirected` | `Y` | `ASSOCIATED_WITH` |
| `스승` | `HAS_TEACHER` | `SOCIAL_TEACHER` | `person_to_related` | `N` | `HAS_STUDENT` |
| `제자` | `HAS_STUDENT` | `SOCIAL_STUDENT` | `person_to_related` | `N` | `HAS_TEACHER` |
| `아내` | `HAS_WIFE` | `SPOUSE` | `person_to_related` | `N` | `HAS_HUSBAND` |
| `남편` | `HAS_HUSBAND` | `SPOUSE` | `person_to_related` | `N` | `HAS_WIFE` |

이 사전이 있으면 전처리 코드에서 `relation_type`별 판단을 계속 `if/elif`로 하드코딩하지 않아도 된다. 사전을 조인해서 관계 의미를 붙이면 된다.

또한 Neo4j에서는 MVP 기준으로 모든 인물 관계를 `RELATED_TO` 하나로 넣더라도 속성으로 의미를 유지할 수 있다.

```text
(:Person)-[:RELATED_TO {
  raw_relation_type: "부",
  normalized_relation_type: "HAS_FATHER",
  relation_group: "FAMILY_PARENT",
  is_symmetric: false,
  inverse_relation_type: "HAS_CHILD"
}]->(:Person)
```

이렇게 하면 가족 관계, 동세대 관계, 사회 관계를 쿼리에서 쉽게 구분할 수 있다.

### 9. `source_url_dictionary.csv`

RAG와 출처 추적을 위한 URL 사전이다.

원본 데이터에는 여러 URL 컬럼이 있다.

```text
itkc_events.detail_url
itkc_event_relations.detail_url
itkc_person_relations.evidence_url
itkc_person_relations.detail_url
```

URL은 그래프 구조 자체에는 필수는 아니지만, 답변의 근거와 RAG 품질에는 중요하다. 특히 Tavily를 이용해 URL 본문을 추출하려면 먼저 중복 제거된 URL 목록이 필요하다.

이 사전이 필요한 이유는 다음과 같다.

- 중복 URL을 제거한다.
- URL이 어떤 테이블/컬럼에서 왔는지 추적한다.
- Tavily extract 대상 URL queue로 쓸 수 있다.
- `Evidence`, `Source`, `DocumentChunk` 확장에 사용할 수 있다.
- 챗봇 답변에 출처 URL을 붙일 수 있다.

핵심 컬럼은 다음과 같다.

| 컬럼 | 의미 |
|---|---|
| `source_url_id` | URL 고유 ID |
| `url` | 원본 URL |
| `source_table` | URL이 나온 테이블 |
| `source_column` | URL이 나온 컬럼 |
| `source_type` | `EVIDENCE`, `DETAIL`, `SOURCE` 등 |
| `use_for_rag` | RAG 수집 대상 여부 |
| `fetch_status` | Tavily 수집 상태 |
| `note` | 비고 |

### 10. 전체 흐름 정리

최종적으로 각 파일은 다음 역할을 맡는다.

| 파일 | 위치 | 역할 |
|---|---|---|
| `category_dictionary.csv` | `dictionary/` | `Category` 노드 기준 사전 |
| `term_category_relation.csv` | `staging/` | `Term - HAS_CATEGORY - Category` 관계 생성용 |
| `event_category_dictionary.csv` | `dictionary/` | 이벤트 원본 분류 사전 |
| `event_category_relation.csv` | `staging/` | `Event - HAS_EVENT_CATEGORY - EventCategory` 관계 생성용 |
| `category_mapping.csv` | `dictionary/` 또는 `mapping/` | 이벤트 분류와 표준 카테고리 연결 규칙 |
| `period_dictionary.csv` | `dictionary/` | `Period` 노드 기준 사전 |
| `event_date_parse.csv` | `staging/` | Event 날짜 정규화 결과 |
| `relation_type_dictionary.csv` | `dictionary/` | 인물 관계 의미/방향/대칭성 규칙 |
| `source_url_dictionary.csv` | `dictionary/` | URL 출처/RAG 수집 기준 사전 |

정리하면 dictionary는 표준 기준을 만들고, staging은 원본을 그래프 관계로 넣기 좋게 펼치며, mapping은 서로 다른 기준을 연결한다. 이 구분을 해두면 Neo4j import, 검수, RAG 확장을 모두 같은 흐름 안에서 관리할 수 있다.
